# 🚇 VBB Transit Data — Exploratory Data Analysis
**Project:** VBB Transit Data Engineering and Analytics Platform  
**Dataset:** Berlin-Brandenburg GTFS (301,078 rows across 8 tables)  
**Database:** PostgreSQL — vbb_db  
**Team:** Pradeep | Abhishek | Mahesh | Nisarga  
**Professor:** Dr. Farhan Khan

---
### Business Questions We Answer Here
- **BQ2** — Which stops and corridors are the busiest?
- **BQ3** — How does service reliability change across the week?
- **BQ5** — How have schedules changed over time?
- **BQ6** — Accessibility analysis across the network

---
## 0. Setup — Import Libraries & Connect to PostgreSQL

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap, MarkerCluster
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# Connect to PostgreSQL
engine = create_engine('postgresql://localhost/vbb_db')
print('✅ Connected to PostgreSQL — vbb_db')
print('📊 Ready to analyse 301,078 rows of VBB transit data')

✅ Connected to PostgreSQL — vbb_db
📊 Ready to analyse 301,078 rows of VBB transit data


---
## 1. Network Overview — What Does the VBB Network Look Like?

In [4]:
# Quick summary of the entire network
summary_queries = {
    'Total Agencies':    'SELECT COUNT(*) as n FROM agency',
    'Total Routes':      'SELECT COUNT(*) as n FROM routes',
    'Total Stops':       'SELECT COUNT(*) as n FROM stops',
    'Total Transfers':   'SELECT COUNT(*) as n FROM transfers',
    'Service Schedules': 'SELECT COUNT(*) as n FROM calendar',
    'Service Exceptions':'SELECT COUNT(*) as n FROM calendar_dates',
    'Station Pathways':  'SELECT COUNT(*) as n FROM pathways',
}

print('🚇 VBB Network — Complete Overview')
print('='*40)
for label, query in summary_queries.items():
    n = pd.read_sql(query, engine).iloc[0]['n']
    print(f'  {label:<22} {n:>8,}')
print('='*40)

🚇 VBB Network — Complete Overview
  Total Agencies               35
  Total Routes              1,322
  Total Stops              41,840
  Total Transfers          52,682
  Service Schedules         3,327
  Service Exceptions       75,729
  Station Pathways        125,940


---
## 2. Transport Mode Analysis — Bus vs Tram vs S-Bahn vs U-Bahn

In [5]:
# How many routes per transport mode?
modes = pd.read_sql("""
    SELECT transport_mode, COUNT(*) as route_count
    FROM routes
    GROUP BY transport_mode
    ORDER BY route_count DESC
""", engine)

print('🚌 Routes by Transport Mode')
print(modes.to_string(index=False))

🚌 Routes by Transport Mode
 transport_mode  route_count
            Bus         1138
Rail (Regional)           69
           Tram           50
         S-Bahn           48
         U-Bahn            9
Water Transport            8


In [6]:
# Pie chart — transport mode breakdown
fig = px.pie(
    modes,
    values='route_count',
    names='transport_mode',
    title='🚌 VBB Network — Routes by Transport Mode',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.update_layout(title_font_size=18, height=500)
fig.show()

In [7]:
# Bar chart — routes per agency (top 15)
agency_routes = pd.read_sql("""
    SELECT a.agency_name, COUNT(r.route_id) as route_count
    FROM agency a
    JOIN routes r ON a.agency_id = r.agency_id
    GROUP BY a.agency_name
    ORDER BY route_count DESC
    LIMIT 15
""", engine)

fig2 = px.bar(
    agency_routes,
    x='route_count',
    y='agency_name',
    orientation='h',
    title='🏢 Top 15 Agencies by Number of Routes',
    labels={'route_count': 'Number of Routes', 'agency_name': 'Agency'},
    color='route_count',
    color_continuous_scale='Blues'
)
fig2.update_layout(
    height=550,
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='white',
    title_font_size=18
)
fig2.show()

---
## 3. BQ2 — Busiest Stops & Transfer Hubs

In [8]:
# Top 15 busiest transfer hubs
top_hubs = pd.read_sql("""
    SELECT
        s.stop_name,
        s.location_type_name,
        s.is_accessible,
        COUNT(t.from_stop_id) AS transfer_count
    FROM stops s
    LEFT JOIN transfers t ON s.stop_id::text = t.from_stop_id::text
    GROUP BY s.stop_name, s.location_type_name, s.is_accessible
    ORDER BY transfer_count DESC
    LIMIT 15
""", engine)

print('🔀 Top 15 Busiest Transfer Hubs in Berlin-Brandenburg')
print(top_hubs.to_string(index=False))

🔀 Top 15 Busiest Transfer Hubs in Berlin-Brandenburg
                            stop_name location_type_name  is_accessible  transfer_count
         S+U Rathaus Spandau (Berlin)    Stop / Platform          False            1177
            Zehlendorf Eiche (Berlin)    Stop / Platform          False             516
              Stahnsdorf, Waldschänke    Stop / Platform          False             304
                Fürstenwalde, Bahnhof    Stop / Platform          False             264
              Cottbus, Stadtpromenade    Stop / Platform          False             226
                Cottbus, Hauptbahnhof    Stop / Platform          False             185
                   S Marzahn (Berlin)    Stop / Platform          False             182
Cottbus, Stadthalle Puschkinpromenade    Stop / Platform           True             177
                   Seelow, Busbahnhof    Stop / Platform          False             155
                    S Hennigsdorf Bhf    Stop / Platform          F

In [9]:
# Horizontal bar chart — top 15 hubs coloured by accessibility
fig3 = px.bar(
    top_hubs,
    x='transfer_count',
    y='stop_name',
    orientation='h',
    color='is_accessible',
    color_discrete_map={True: '#2ecc71', False: '#e74c3c'},
    title='🔀 Top 15 Busiest Transfer Hubs (Green = Wheelchair Accessible)',
    labels={
        'transfer_count': 'Number of Transfers',
        'stop_name': 'Stop Name',
        'is_accessible': 'Wheelchair Accessible'
    }
)
fig3.update_layout(
    height=550,
    yaxis={'categoryorder': 'total ascending'},
    plot_bgcolor='white',
    title_font_size=16
)
fig3.show()

---
## 4. Accessibility Analysis — How Inclusive is the VBB Network?

In [10]:
# Accessibility breakdown by stop type
accessibility = pd.read_sql("""
    SELECT
        is_accessible,
        location_type_name,
        COUNT(*) as stop_count
    FROM stops
    GROUP BY is_accessible, location_type_name
    ORDER BY stop_count DESC
""", engine)

# Overall accessibility percentage
total      = pd.read_sql('SELECT COUNT(*) as n FROM stops', engine).iloc[0]['n']
accessible = pd.read_sql('SELECT COUNT(*) as n FROM stops WHERE is_accessible = true', engine).iloc[0]['n']
pct        = round(accessible / total * 100, 1)

print(f'♿ Overall Accessibility Summary')
print(f'   Total stops       : {total:,}')
print(f'   Accessible stops  : {accessible:,}')
print(f'   Accessibility rate: {pct}%')
print(f'   Not accessible    : {total - accessible:,} stops need improvement')

♿ Overall Accessibility Summary
   Total stops       : 41,840
   Accessible stops  : 3,235
   Accessibility rate: 7.7%
   Not accessible    : 38,605 stops need improvement


In [11]:
# Grouped bar chart — accessibility by stop type
fig4 = px.bar(
    accessibility,
    x='location_type_name',
    y='stop_count',
    color='is_accessible',
    barmode='group',
    color_discrete_map={True: '#2ecc71', False: '#e74c3c'},
    title='♿ Wheelchair Accessibility by Stop Type',
    labels={
        'stop_count': 'Number of Stops',
        'location_type_name': 'Stop Type',
        'is_accessible': 'Accessible'
    }
)
fig4.update_layout(plot_bgcolor='white', title_font_size=16, height=450)
fig4.show()

---
## 5. BQ3 — Service Reliability: How Does It Change Across the Week?

In [12]:
# How many services run on each day of the week?
day_service = pd.read_sql("""
    SELECT
        SUM(monday)    AS Monday,
        SUM(tuesday)   AS Tuesday,
        SUM(wednesday) AS Wednesday,
        SUM(thursday)  AS Thursday,
        SUM(friday)    AS Friday,
        SUM(saturday)  AS Saturday,
        SUM(sunday)    AS Sunday
    FROM calendar
""", engine)

# Reshape for plotting
day_df = day_service.T.reset_index()
day_df.columns = ['day', 'service_count']

# Keep correct day order
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_df['day'] = pd.Categorical(day_df['day'], categories=day_order, ordered=True)
day_df = day_df.sort_values('day')

print('📅 Services Running Per Day of Week')
print(day_df.to_string(index=False))

📅 Services Running Per Day of Week
day  service_count
NaN          786.0
NaN          806.0
NaN          806.0
NaN          775.0
NaN          794.0
NaN          682.0
NaN          644.0


In [13]:
# Line chart — service frequency across the week
fig5 = px.bar(
    day_df,
    x='day',
    y='service_count',
    title='📅 BQ3 — How Many Services Run Each Day of the Week?',
    labels={'service_count': 'Number of Services', 'day': 'Day of Week'},
    color='service_count',
    color_continuous_scale='Blues'
)
fig5.update_layout(plot_bgcolor='white', title_font_size=16, height=450)
fig5.show()

In [14]:
# Service pattern breakdown — Full Week vs Weekdays Only vs Weekends Only
patterns = pd.read_sql("""
    SELECT service_pattern, COUNT(*) as count
    FROM calendar
    GROUP BY service_pattern
    ORDER BY count DESC
""", engine)

fig6 = px.pie(
    patterns,
    values='count',
    names='service_pattern',
    title='📅 Service Pattern Distribution (Full Week / Weekdays / Weekends)',
    color_discrete_sequence=['#3498db','#2ecc71','#e74c3c','#f39c12'],
    hole=0.3
)
fig6.update_traces(textposition='inside', textinfo='percent+label')
fig6.update_layout(title_font_size=16, height=450)
fig6.show()

---
## 6. BQ5 — Historical Schedule Changes: Service Added vs Removed

In [15]:
# Service exceptions by month — added vs removed
exceptions = pd.read_sql("""
    SELECT month, year, exception_label, COUNT(*) as count
    FROM calendar_dates
    GROUP BY month, year, exception_label
    ORDER BY year, count DESC
""", engine)

print('📆 Service Exceptions by Month')
print(exceptions.to_string(index=False))

📆 Service Exceptions by Month
    month  year exception_label  count
      May  2026   Service Added  18426
     June  2026   Service Added  13933
      May  2026 Service Removed   9223
     July  2026   Service Added   5172
  October  2026 Service Removed   4883
     June  2026 Service Removed   3523
 November  2026 Service Removed   3213
  October  2026   Service Added   2344
 November  2026   Service Added   2306
   August  2026   Service Added   2237
     July  2026 Service Removed   2135
   August  2026 Service Removed   1911
September  2026   Service Added   1720
 December  2026 Service Removed   1520
September  2026 Service Removed   1430
 December  2026   Service Added    835
    April  2026   Service Added    629
    April  2026 Service Removed    289


In [16]:
# Grouped bar chart — service added vs removed per month
fig7 = px.bar(
    exceptions,
    x='month',
    y='count',
    color='exception_label',
    barmode='group',
    color_discrete_map={
        'Service Added':   '#2ecc71',
        'Service Removed': '#e74c3c'
    },
    title='📆 BQ5 — Service Added vs Removed by Month',
    labels={
        'count': 'Number of Exceptions',
        'month': 'Month',
        'exception_label': 'Exception Type'
    }
)
fig7.update_layout(plot_bgcolor='white', title_font_size=16, height=450)
fig7.show()

In [17]:
# Which day of the week sees the most service exceptions?
day_exceptions = pd.read_sql("""
    SELECT day_of_week, exception_label, COUNT(*) as count
    FROM calendar_dates
    GROUP BY day_of_week, exception_label
    ORDER BY count DESC
""", engine)

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_exceptions['day_of_week'] = pd.Categorical(
    day_exceptions['day_of_week'], categories=day_order, ordered=True
)
day_exceptions = day_exceptions.sort_values('day_of_week')

fig8 = px.bar(
    day_exceptions,
    x='day_of_week',
    y='count',
    color='exception_label',
    barmode='group',
    color_discrete_map={
        'Service Added':   '#2ecc71',
        'Service Removed': '#e74c3c'
    },
    title='📆 Service Exceptions by Day of Week',
    labels={
        'count': 'Number of Exceptions',
        'day_of_week': 'Day',
        'exception_label': 'Exception Type'
    }
)
fig8.update_layout(plot_bgcolor='white', title_font_size=16, height=450)
fig8.show()

---
## 7. Station Infrastructure — Pathways Inside Stations

In [18]:
# What types of pathways exist inside VBB stations?
pathways = pd.read_sql("""
    SELECT pathway_mode_name, direction, COUNT(*) as count
    FROM pathways
    GROUP BY pathway_mode_name, direction
    ORDER BY count DESC
""", engine)

print('🚶 Station Pathway Types')
print(pathways.to_string(index=False))

🚶 Station Pathway Types
pathway_mode_name direction  count
          Walkway   One-Way 119231
           Stairs   One-Way   4410
         Elevator   One-Way   1544
        Escalator   One-Way    755


In [19]:
# Sunburst chart — pathway types and directions
pathway_summary = pd.read_sql("""
    SELECT pathway_mode_name, COUNT(*) as count
    FROM pathways
    GROUP BY pathway_mode_name
    ORDER BY count DESC
""", engine)

fig9 = px.bar(
    pathway_summary,
    x='pathway_mode_name',
    y='count',
    title='🚶 Station Infrastructure — Pathway Types Across VBB Network',
    labels={'count': 'Number of Pathways', 'pathway_mode_name': 'Pathway Type'},
    color='count',
    color_continuous_scale='Viridis'
)
fig9.update_layout(plot_bgcolor='white', title_font_size=16, height=450)
fig9.show()

---
## 8. Interactive Berlin Transit Map

In [20]:
# Load stations from PostgreSQL
stations = pd.read_sql("""
    SELECT stop_name, stop_lat, stop_lon,
           location_type_name, is_accessible
    FROM stops
    WHERE location_type_name = 'Station'
    LIMIT 800
""", engine)

print(f'Loaded {len(stations):,} stations for map')

# Create Berlin map
m = folium.Map(
    location=[52.5200, 13.4050],
    zoom_start=11,
    tiles='CartoDB positron'
)

# Add marker cluster for better performance
cluster = MarkerCluster().add_to(m)

# Add each station
for _, row in stations.iterrows():
    color = 'green' if row['is_accessible'] else 'red'
    folium.CircleMarker(
        location=[row['stop_lat'], row['stop_lon']],
        radius=6,
        color=color,
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{row['stop_name']}</b><br>"
            f"Type: {row['location_type_name']}<br>"
            f"Accessible: {'Yes ♿' if row['is_accessible'] else 'No'}",
            max_width=200
        ),
        tooltip=row['stop_name']
    ).add_to(cluster)

# Legend
legend = '''
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px; border-radius:8px;
     box-shadow:2px 2px 6px rgba(0,0,0,0.3); font-family:Arial">
  <b>🚇 VBB Stations</b><br>
  <span style="color:green">●</span> Wheelchair Accessible<br>
  <span style="color:red">●</span> Not Accessible
</div>
'''
m.get_root().html.add_child(folium.Element(legend))

# Save map
map_path = '/Users/abhishekkarthikakunuru/Desktop/Data Engineering /berlin_stations_map.html'
m.save(map_path)
print(f'✅ Map saved! Open in your browser:')
print(f'   {map_path}')
m

Loaded 800 stations for map
✅ Map saved! Open in your browser:
   /Users/abhishekkarthikakunuru/Desktop/Data Engineering /berlin_stations_map.html


In [ ]:
# Heatmap — where are stops most dense?
all_stops = pd.read_sql("""
    SELECT stop_lat, stop_lon
    FROM stops
    WHERE stop_lat IS NOT NULL
""", engine)

heatmap = folium.Map(
    location=[52.5200, 13.4050],
    zoom_start=10,
    tiles='CartoDB dark_matter'
)

heat_data = all_stops[['stop_lat', 'stop_lon']].values.tolist()
HeatMap(heat_data, radius=8, blur=10, min_opacity=0.4).add_to(heatmap)

heatmap_path = '/Users/abhishekkarthikakunuru/Desktop/Data Engineering /berlin_heatmap.html'
heatmap.save(heatmap_path)
print(f'✅ Heatmap saved!')
print(f'   {heatmap_path}')
heatmap

---
## 9. Summary — Key Findings

In [ ]:
# Final summary of all key findings
total_stops  = pd.read_sql('SELECT COUNT(*) as n FROM stops', engine).iloc[0]['n']
accessible   = pd.read_sql('SELECT COUNT(*) as n FROM stops WHERE is_accessible = true', engine).iloc[0]['n']
total_routes = pd.read_sql('SELECT COUNT(*) as n FROM routes', engine).iloc[0]['n']
top_hub      = pd.read_sql("""
    SELECT s.stop_name, COUNT(t.from_stop_id) as cnt
    FROM stops s
    LEFT JOIN transfers t ON s.stop_id::text = t.from_stop_id::text
    GROUP BY s.stop_name ORDER BY cnt DESC LIMIT 1
""", engine).iloc[0]
busiest_day  = pd.read_sql("""
    SELECT day_of_week, COUNT(*) as cnt
    FROM calendar_dates WHERE exception_label = 'Service Removed'
    GROUP BY day_of_week ORDER BY cnt DESC LIMIT 1
""", engine).iloc[0]

print('='*55)
print('  📊 VBB EDA — KEY FINDINGS SUMMARY')
print('='*55)
print(f'  Total stops in network   : {total_stops:,}')
print(f'  Wheelchair accessible    : {accessible:,} ({round(accessible/total_stops*100,1)}%)')
print(f'  NOT accessible           : {total_stops-accessible:,} ({round((total_stops-accessible)/total_stops*100,1)}%)')
print(f'  Total routes             : {total_routes:,}')
print(f'  Busiest transfer hub     : {top_hub["stop_name"]} ({top_hub["cnt"]:,} connections)')
print(f'  Most disrupted day       : {busiest_day["day_of_week"]} (most service removals)')
print('='*55)
print('  Business Questions Answered:')
print('  ✅ BQ2 — Busiest stops identified (transfer hub analysis)')
print('  ✅ BQ3 — Service patterns across weekdays vs weekends')
print('  ✅ BQ5 — Historical exceptions tracked by month/day')
print('  ✅ BQ6 — PostgreSQL is the query-ready analytical surface')
print('='*55)